<a href="https://colab.research.google.com/github/gayakarapetyan/PythonCourseH2/blob/main/Session4/RecapandNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 1: Recap

---
## Section 1 - Writing and reading a file '

In [ ]:
from pathlib import Path

sample_dir = Path("class_files")
sample_dir.mkdir(exist_ok=True)

In [ ]:
# --- write a simple text file
with open(sample_dir / "test.txt", "w") as f:
    f.write("Hello!\n")
    f.write("This is line two.\n")

In [ ]:
# --- read it
with open(sample_dir / "test.txt", "r") as f:
    content = f.read()
print("\nfile content:\n" + content)

---
## Section 2 - Reading different file formats

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

# Reuse the same folder from Section 1
sample_dir = Path("class_files")
sample_dir.mkdir(exist_ok=True)

# Some random generated weather measurements
rng = np.random.default_rng(0)
df_demo = pd.DataFrame({
    "date": pd.date_range("2024-01-01", periods=10, freq="D"),
    "station": ["Magdeburg"] * 10,
    "temp_c": np.round(rng.normal(5, 3, 10), 1),
    "rain_mm": np.round(np.abs(rng.normal(2, 2, 10)), 1),
})

# Save it in three different formats.
df_demo.to_csv(sample_dir / "weather.csv", index=False)
df_demo.to_excel(sample_dir / "weather.xlsx", index=False)
df_demo.to_json(sample_dir / "weather.json", orient="records", date_format="iso")

print("Created:", [p.name for p in sample_dir.iterdir()])

### CSV format

`pd.read_csv`

- `sep` - the column separator (`,` by default, but many use `;`).
- `parse_dates` - turn a text column into real dates so you can sort and resample by time.
- `na_values` - tell pandas which strings (e.g. `"-999"`) actually mean "missing".

In [ ]:
weather = pd.read_csv(sample_dir / "weather.csv", parse_dates=["date"])
print(weather.dtypes)   # 'date' is now datetime64, not text
weather.head()

### Excel format

function: `pd.read_excel`. A workbook can have several sheets, so `sheet_name` to pick one. (Excel support comes from the `openpyxl` package)

In [ ]:
weather_xl = pd.read_excel(sample_dir / "weather.xlsx", sheet_name=0)
weather_xl.head()

### JSON format

- `pd.read_json` when it is already table-shaped.
- the built-in `json` module when you need to look inside the structure first.

In [ ]:
# Table-shaped JSON to a DataFrame
weather_js = pd.read_json(sample_dir / "weather.json")
print("As a table:")
print(weather_js.head(2))

# The raw structure (this is what an API actually hands you)
with open(sample_dir / "weather.json") as f:
    raw = json.load(f)
print("\nFirst record as a Python dict:")
print(raw[0])

### netCDF and an HDF5 file

 Below we build a *tiny* gridded temperature dataset - 3 days x 3 latitudes x 3 longitudes - save it as **netCDF**, and read it back.

*(On Colab these are usually already installed. If not, run `!pip install xarray netcdf4 h5py`.)*

In [ ]:
import xarray as xr

# A gridded dataset: temperature over (time, lat, lon)
grid_rng = np.random.default_rng(1)
times = pd.date_range("2024-01-01", periods=3)
lats = [52.0, 52.5, 53.0]
lons = [11.0, 11.5, 12.0]
grid_temp = (10 + grid_rng.normal(0, 3, size=(3, 3, 3))).round(1)

ds = xr.Dataset(
    {"t2m": (["time", "lat", "lon"], grid_temp)},
    coords={"time": times, "lat": lats, "lon": lons},  # the labels for each axis
)
ds["t2m"].attrs["units"] = "degC"                       # metadata travels inside the file

ds.to_netcdf(sample_dir / "mini_climate.nc")            # save
reloaded = xr.open_dataset(sample_dir / "mini_climate.nc")  # read back
print(reloaded)

# Grab one grid point across all days by its real coordinates
point = reloaded["t2m"].sel(lat=52.5, lon=11.5)
print("\nTemperature at (52.5N, 11.5E) over the 3 days:", point.values)

In [ ]:
import h5py

with h5py.File(sample_dir / "mini.h5", "w") as f:
    dset = f.create_dataset("temperature", data=grid_temp)
    dset.attrs["units"] = "degC"

with h5py.File(sample_dir / "mini.h5", "r") as f:
    arr = f["temperature"][:]            # [:] pulls the stored data into a numpy array
    print("HDF5 dataset 'temperature':", arr.shape, "| units:", f["temperature"].attrs["units"])

### Your turn - read a TSV file

Many data exports are *tab*-separated, not comma-separated. Save the weather table as a `.tsv`, then read it back.

you have to use `to_csv` and `read_csv` functions

In [ ]:
#your code here

<details>
<summary><b>Show a solution</b></summary>

```python
weather.to_csv(sample_dir / "weather.tsv", sep="\t", index=False)
tsv_back = pd.read_csv(sample_dir / "weather.tsv", sep="\t")   
tsv_back.head()
```

</details>

---
## Section 3 - Processing data

In [ ]:
# generating a year of daily synthetic weather
days = pd.date_range("2024-01-01", "2024-12-31", freq="D")
n = len(days)
seasonal = 10 - 12 * np.cos(2 * np.pi * days.dayofyear / 365)
temp = seasonal + rng.normal(0, 2.5, n)
rain = np.abs(rng.normal(1.5, 2.0, n))

data = pd.DataFrame({"date": days, "temp_c": temp.round(1), "rain_mm": rain.round(1)})

# adding a few missing values on purpose
missing_idx = rng.choice(n, 8, replace=False)
data.loc[missing_idx, "temp_c"] = np.nan

data.head()

Try: `head()`, `info()`, `describe()`. `info()` . gives you description of the data

In [ ]:
data.info()

Select and filter. Pick columns with `data["col"]`, pick rows with a boolean condition. Combine conditions with `&` (and) / `|` (or), each part in parentheses.

In [ ]:
# All the rainy summer days
summer = data[(data["date"].dt.month.isin([6, 7, 8])) & (data["rain_mm"] > 4)]
print(f"{len(summer)} reiny summer days")
summer.head()

In [ ]:
# minitask: take rainy spring nter days and print the siye and description

**Missing values.** Either drop them or fill them.
filling with the value before/after (`interpolate`) is usually used, because it keeps the calendar continuous.

In [ ]:
print("Missing before:", data["temp_c"].isna().sum())
data["temp_c"] = data["temp_c"].interpolate()
print("Missing after: ", data["temp_c"].isna().sum())

**Group and aggregate.**

In [ ]:
monthly = data.groupby(data["date"].dt.month).agg(
    mean_temp=("temp_c", "mean"),
    total_rain=("rain_mm", "sum"),
).round(1)
monthly.index.name = "month"
print(monthly)

**Plot**

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
ax[0].plot(data["date"], data["temp_c"], color="#1F4E79")
ax[0].set_ylabel("Temp (C)")
ax[0].set_title("A generated data in a year in Magdeburg")
ax[1].bar(data["date"], data["rain_mm"], color="#4C72B0", width=1.0)
ax[1].set_ylabel("Rain (mm)")
plt.tight_layout()
plt.show()

### Your turn - warmest days and rain per season

Using the `data` DataFrame: (a) the 5 warmest days are found for you; (b) compute the **total rainfall per season** yourself (DJF = Dec/Jan/Feb, MAM = Mar/Apr/May, and so on).

In [ ]:
# (a)  the 5 warmest days:
warmest = data.sort_values("temp_c", ascending=False).head(5)
print(warmest[["date", "temp_c"]].to_string(index=False))

# (b) total rainfall per season.
# you can use map function to map each month number to a season label, then groupby it and sum "rain_mm".

#just a reminder about map function
def part_of_day(hour):
    if hour < 12:
        return "Morning"
    elif hour < 18:
        return "Afternoon"
    else:
        return "Evening"

hours = pd.Series([7, 9, 13, 16, 20, 23])
hours.map(part_of_day)

<details>
<summary><b>Show a solution</b></summary>

```python
month_to_season = {}
for m in [12, 1, 2]:  month_to_season[m] = "Winter"
for m in [3, 4, 5]:   month_to_season[m] = "Spring"
for m in [6, 7, 8]:   month_to_season[m] = "Summer"
for m in [9, 10, 11]: month_to_season[m] = "Autumn"

data["season"] = data["date"].dt.month.map(month_to_season)
```

or
```python
season = data["date"].dt.month.map({12:"DJF",1:"DJF",2:"DJF",
                                    3:"MAM",4:"MAM",5:"MAM",
                                    6:"JJA",7:"JJA",8:"JJA",
                                    9:"SON",10:"SON",11:"SON"})
print(data.groupby(season)["rain_mm"].sum().round(1))
```

</details>

---
## Section 4 - Neural networks

> **A neural network is a flexible function that bends itself to fit your data.**

Linear regression fits a straight line. A neural network fits *curved, * relationships by stacking lots of tiny simple pieces.

### One neuron

A single neuron does two things:
1. a **weighted sum** of its inputs plus a bias - `z = w1*x1 + w2*x2 + ... + b`. That part is *exactly* linear regression.
2. an **activation function** applied to that sum - `a = f(z)`.

Stack neurons into **layers**, stack layers into a **network**. Two knobs describe the shape:
- **width** = neurons per layer (how much detail one layer can capture),
- **depth** = number of layers (how many transformations in chain).

In [ ]:
# Synthetic non-linear data, by knowing the true shape, so we can judge the model.
X = np.linspace(0, 1, 200).reshape(-1, 1)
true = np.sin(2.5 * np.pi * X).ravel()          # used to return a contiguous array. This function returns a 1D array that contains the input elements.
y = true + rng.normal(0, 0.15, size=true.shape) # what we "measure" (+nois)

plt.figure(figsize=(8, 4))
plt.scatter(X, y, s=12, color="#4C72B0", label="noisy measurements")
plt.plot(X, true, color="#1F4E79", lw=2, label="true relationship")
plt.legend(); plt.title("Our synthetic problem"); plt.show()

Before any neural network, fit a straight line.

In [ ]:
from sklearn.linear_model import LinearRegression

lin = LinearRegression().fit(X, y)

plt.figure(figsize=(8, 4))
plt.scatter(X, y, s=12, color="#4C72B0", alpha=0.6)
plt.plot(X, lin.predict(X), color="crimson", lw=2, label="straight line")
plt.plot(X, true, color="#1F4E79", lw=1, ls="--", label="truth")
plt.legend(); plt.title("Linear regression underfits a curve"); plt.show()

We use scikit-learn's `MLPRegressor` where the key argument is `activation`.

Watch what happens when we use a **linear** activation (`activation="identity"`)

In [ ]:
from sklearn.neural_network import MLPRegressor

# A BIG network (3 layers x 64 neurons) but with a LINEAR activation
linear_nn = MLPRegressor(hidden_layer_sizes=(64, 64, 64),
                         activation="identity",   # linear function
                         max_iter=2000, random_state=0).fit(X, y)

plt.figure(figsize=(8, 4))
plt.scatter(X, y, s=12, color="#4C72B0", alpha=0.6)
plt.plot(X, linear_nn.predict(X), color="crimson", lw=2, label="deep network, LINEAR activation")
plt.plot(X, true, color="#1F4E79", lw=1, ls="--", label="truth")
plt.legend(); plt.title("192 neurons... still a straight line"); plt.show()

> Without a non-linear activation, stacking layers is pointless - a chain of linear steps is just one bigger linear step. The activation function is the *entire reason* a deep network can do more than linear regression.

Now swap in `relu` (the modern default) and change nothing else:

In [ ]:
relu_nn = MLPRegressor(hidden_layer_sizes=(64, 64, 64),
                       activation="relu",      # <-- now it can bend
                       max_iter=2000, random_state=0).fit(X, y)

plt.figure(figsize=(8, 4))
plt.scatter(X, y, s=12, color="#4C72B0", alpha=0.6)
plt.plot(X, relu_nn.predict(X), color="crimson", lw=2, label="same network, ReLU activation")
plt.plot(X, true, color="#1F4E79", lw=1, ls="--", label="truth")
plt.legend(); plt.title("One word changed (identity -> relu) and it fits the curve"); plt.show()

### Your turn - try a different activation

We saw `identity` and `relu` that works. Fit the curve with `activation='tanh'` and just **8** neurons in a single layer. then change the number of neutons.

In [ ]:
#before lets try with relu
m = MLPRegressor(hidden_layer_sizes=(8,), activation="relu",   # <- change "relu" to "tanh"
                 solver="lbfgs", max_iter=3000, random_state=0).fit(X, y)

plt.figure(figsize=(8, 4))
plt.scatter(X, y, s=12, color="#4C72B0", alpha=0.6)
plt.plot(X, m.predict(X), color="crimson", lw=2, label="your network")
plt.plot(X, true, color="#1F4E79", lw=1, ls="--", label="truth")
plt.legend(); plt.show()

<details>
<summary><b>Show a solution</b></summary>

```python
m = MLPRegressor(hidden_layer_sizes=(8,), activation="tanh",
                 solver="lbfgs", max_iter=3000, random_state=0).fit(X, y)

plt.scatter(X, y, s=12, alpha=0.6)
plt.plot(X, m.predict(X), color="crimson", lw=2)
plt.plot(X, true, ls="--")
plt.show()
```

</details>

### Step 3 - what the different activation functions look like

`relu`, `tanh`, and `logistic` (sigmoid) are the three you will meet. They fit this curve a little differently - `relu` gives slightly angular fits, `tanh` smooth ones. For hidden layers, **`relu` is the  default**; the others are mostly historical or for special cases.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharey=True)
for ax, act in zip(axes, ["relu", "tanh", "logistic"]):
    m = MLPRegressor(hidden_layer_sizes=(32, 32), activation=act,
                     solver="lbfgs", max_iter=3000, random_state=0).fit(X, y)
    ax.scatter(X, y, s=8, color="#4C72B0", alpha=0.5)
    ax.plot(X, m.predict(X), color="crimson", lw=2)
    ax.plot(X, true, color="#1F4E79", lw=1, ls="--")
    ax.set_title(f"activation = '{act}'")
plt.tight_layout(); plt.show()

### Step 4 - width and depth

Now we change the *architecture* and watch underfitting and overfitting appear.

In [ ]:
# change slightly the training data so a big network can overfit .
rng_demo = np.random.default_rng(0)
X_few = np.linspace(0, 1, 25).reshape(-1, 1)
y_few = np.sin(2.5 * np.pi * X_few).ravel() + rng_demo.normal(0, 0.15, size=25)

X_grid = np.linspace(0, 1, 400).reshape(-1, 1)
true_grid = np.sin(2.5 * np.pi * X_grid).ravel()

configs = [(2,), (26,), (300, 300)]
titles = ["Too small (underfit)", "About right", "Too big (overfit)"]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharey=True)
for ax, cfg, title in zip(axes, configs, titles):
    m = MLPRegressor(hidden_layer_sizes=cfg, activation="relu",
                     solver="lbfgs", max_iter=5000, random_state=0).fit(X_few, y_few)
    ax.scatter(X_few, y_few, s=20, color="#4C72B0", alpha=0.7)      # the 25 training points
    ax.plot(X_grid, m.predict(X_grid), color="crimson", lw=2)        # learned curve (dense)
    ax.plot(X_grid, true_grid, color="#1F4E79", lw=1, ls="--")       # the truth
    ax.set_title(f"{title}\nhidden_layer_sizes={cfg}")
plt.tight_layout(); plt.show()

###  choosing the output function and the loss: let the task decide

Hidden layers: use `relu`. The **last** layer and the **loss** are dictated by *what you are predicting*. This little table is worth memorising:

| Task | Example | Output activation | Loss function |
|---|---|---|---|
| **Regression** (a number) | temperature, discharge | none / linear | mean squared error (MSE) or MAE |
| **Binary** (yes / no) | flood / no-flood | sigmoid | binary cross-entropy |
| **Multi-class** (pick one of several) | clear / rain / storm | softmax | categorical cross-entropy |


In scikit-learn this choice is made *for* you by which class you pick: `MLPRegressor` (linear output, MSE) vs. `MLPClassifier` (sigmoid/softmax output, cross-entropy). When you move to a deep-learning library you set it by hand.

### The same network, written in Keras


```python
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Dense(64, activation="relu", input_shape=(1,)),  # hidden layer 1
    layers.Dense(64, activation="relu"),                    # hidden layer 2
    layers.Dense(1),                                        # linear output -> regression
])
model.compile(optimizer="adam", loss="mse")                # MSE because it's regression
model.fit(X, y, epochs=200, validation_split=0.3)
```
